# Workshop 4: Eigen Dataset Labelen en Trainen (Notebook)

Welkom! In deze workshop doorloop je **de complete computer vision workflow**:
1. Labelregels afspreken
2. Dataset structureren en valideren
3. Model trainen (zoals in workshop 3)
4. Evalueren en verbeteren
5. Realtime testen

> Deze notebook volgt de labeling best-practices uit `research/labeling-best-practices.md`.


## Leerdoelen

Na deze workshop kun je:
- Een bruikbare object detectie dataset opzetten (train/valid/test)
- Consistent labelen met teamafspraken
- Een YOLO-model trainen op eigen data
- Resultaten interpreteren (precision/recall/mAP)
- Verbeteracties formuleren voor een volgende trainingsronde


## Stap 0 — Setup & Config

Voer eerst de volgende cellen uit. Pas daarna eventueel instellingen aan.


In [ ]:
import os
import sys
import random
import shutil
from pathlib import Path

import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)
print("Setup geladen")


In [ ]:
# ========== CONFIG ========== 
# Zet DATASET_ROOT naar jouw gelabelde dataset map.
# Verwachte structuur:
# dataset/
#   train/images, train/labels
#   valid/images, valid/labels
#   test/images,  test/labels
#   data.yaml

DATASET_ROOT = Path("dataset").resolve()
DATA_YAML = DATASET_ROOT / "data.yaml"

MODEL_SIZE = "yolov8n.pt"
EPOCHS = 40
IMG_SIZE = 640
BATCH = 8
DEVICE = "cpu"   # 0 = eerste NVIDIA GPU
CONF = 0.4

RUN_NAME = "workshop4_eigen_dataset"
PROJECT_DIR = Path("runs/workshop4")

print("DATASET_ROOT:", DATASET_ROOT)
print("DATA_YAML:", DATA_YAML)


## Stap 1 — Teamafspraken voor labelkwaliteit

Gebruik deze checklist **voordat** je gaat labelen:

- Class-definities helder? (wat telt wel/niet mee)
- Minimum zichtbaarheid afgesproken? (bijv. >= 30%)
- Omgang met overlap/occlusie afgesproken?
- Tiny objecten wel/niet labelen afgesproken?
- Minimaal 10-20% dubbel gelabeld voor kwaliteitscheck?

**Denkvraag:** Welke 2 randgevallen zijn voor jullie praktijk het belangrijkst?


In [ ]:
# Vul jullie classnamen in deze lijst (volgorde = class-id)
CLASS_NAMES = [
    "product_ok",
    "product_defect"
]

print("Classes:")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {i}: {c}")


## Stap 2 — Controleer datasetstructuur


In [ ]:
required = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",
    DATASET_ROOT / "valid" / "images",
    DATASET_ROOT / "valid" / "labels",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
    DATA_YAML,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Ontbrekende onderdelen:")
    for m in missing:
        print(" -", m)
else:
    print("Datastructuur is compleet")


In [ ]:
def count_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return len([p for p in folder.iterdir() if p.suffix.lower() in exts])

for split in ["train", "valid", "test"]:
    n = count_images(DATASET_ROOT / split / "images")
    print(f"{split}: {n} images")


## Stap 3 — data.yaml genereren of controleren

Als je al een correcte `data.yaml` hebt, laat deze cel vooral staan als check.


In [ ]:
yaml_data = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print("data.yaml geschreven naar", DATA_YAML)
print(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True))


## Stap 4 — Snelle label-validatie

Controleert o.a.:
- Ontbrekende labelbestanden
- Lege labels
- Class-id buiten bereik
- Waarden buiten [0,1]


In [ ]:
from collections import Counter

problems = []
class_counter = Counter()

for split in ["train", "valid", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            continue

        label_path = lbl_dir / (img_path.stem + ".txt")
        if not label_path.exists():
            problems.append((split, img_path.name, "missing_label"))
            continue

        txt = label_path.read_text(encoding="utf-8").strip()
        if not txt:
            problems.append((split, img_path.name, "empty_label"))
            continue

        for line_no, line in enumerate(txt.splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                problems.append((split, img_path.name, f"bad_format_line_{line_no}"))
                continue

            try:
                cid = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                problems.append((split, img_path.name, f"parse_error_line_{line_no}"))
                continue

            if cid < 0 or cid >= len(CLASS_NAMES):
                problems.append((split, img_path.name, f"invalid_class_{cid}"))
            if not all(0.0 <= v <= 1.0 for v in [x, y, w, h]):
                problems.append((split, img_path.name, f"out_of_range_line_{line_no}"))

            class_counter[cid] += 1

print("Annotaties per class:")
for cid, cnt in sorted(class_counter.items()):
    print(f"  {cid} ({CLASS_NAMES[cid]}): {cnt}")

print("
Aantal problemen:", len(problems))
for p in problems[:20]:
    print(" -", p)
if len(problems) > 20:
    print(f"... en nog {len(problems)-20} meer")


## Stap 5 — Visuele spot-check

Bekijk random voorbeelden met bounding boxes. Gebruik dit om inconsistente labeling snel te zien.


In [ ]:
def yolo_to_xyxy(x, y, w, h, W, H):
    x1 = int((x - w/2) * W)
    y1 = int((y - h/2) * H)
    x2 = int((x + w/2) * W)
    y2 = int((y + h/2) * H)
    return x1, y1, x2, y2

split = "train"
img_dir = DATASET_ROOT / split / "images"
lbl_dir = DATASET_ROOT / split / "labels"
images = [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]

sample = random.sample(images, k=min(6, len(images)))
plt.figure(figsize=(14, 8))

for i, img_path in enumerate(sample, 1):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    label_path = lbl_dir / (img_path.stem + ".txt")
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").strip().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cid = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            x1, y1, x2, y2 = yolo_to_xyxy(x, y, w, h, W, H)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img, CLASS_NAMES[cid], (x1, max(20,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name)

plt.tight_layout()
plt.show()


## Stap 6 — Trainen (zelfde lijn als workshop 3)

Tip: Start met 30-50 epochs. Verhoog pas als val-metrics nog stijgen.


In [ ]:
model = YOLO(MODEL_SIZE)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    patience=20,
)

print("Training klaar")


## Stap 7 — Evaluatie

Hier check je of het model echt geleerd heeft.
Belangrijk: kijk niet alleen naar 1 metric.


In [ ]:
best_weights = PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
print("Best model:", best_weights)

best_model = YOLO(str(best_weights))
val_metrics = best_model.val(data=str(DATA_YAML), split="test", device=DEVICE)
print("Evaluatie klaar")
print(val_metrics)


## Stap 8 — Inference op testbeelden


In [ ]:
test_img_dir = DATASET_ROOT / "test" / "images"
test_imgs = [p for p in test_img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]

sample_test = random.sample(test_imgs, k=min(6, len(test_imgs)))

plt.figure(figsize=(14, 8))
for i, p in enumerate(sample_test, 1):
    pred = best_model.predict(source=str(p), conf=CONF, verbose=False)[0]
    plotted = pred.plot()[:, :, ::-1]  # BGR->RGB

    plt.subplot(2, 3, i)
    plt.imshow(plotted)
    plt.axis("off")
    plt.title(p.name)

plt.tight_layout()
plt.show()


## Stap 9 — Realtime demo

Druk op **q** om te stoppen.


In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Camera kon niet geopend worden")

print("Realtime detectie gestart. Druk q om te stoppen.")
while True:
    ret, frame = cap.read()
    if not ret:
        break

    pred = best_model.predict(source=frame, conf=CONF, verbose=False)[0]
    annotated = pred.plot()
    cv2.imshow("Workshop 4 - Realtime", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Gestopt")


## Stap 10 — Reflectie en verbeterplan

Beantwoord als team:
1. Welke fouten zien we het meest? (false positives / false negatives)
2. Komt dat door data, labels of training-instellingen?
3. Welke 3 acties doen we in ronde 2?

Voorbeelden van acties:
- Meer voorbeelden van randgevallen toevoegen
- Labelregels aanscherpen en opnieuw afstemmen
- Hogere `imgsz` testen voor kleine objecten
- Class-balans verbeteren
